# TACS CIFAR-100 trên Google Colab

Notebook này chạy DINOv3/TACS trên CIFAR-100, lưu checkpoint vào Google Drive và tự resume sau khi Colab ngắt phiên. Bundle mặc định có thể chứa checkpoint local seed 42 để tiếp tục từ epoch đã đạt được.

Trước khi chạy: chọn **Runtime → Change runtime type → GPU**, upload `TACS_Colab_Bundle.zip` vào thư mục gốc My Drive, rồi chạy các cell theo thứ tự.

In [ ]:
from pathlib import Path

# Chỉ cần sửa các giá trị này khi chuyển seed hoặc muốn đổi cấu hình.
BUNDLE_PATH = Path('/content/drive/MyDrive/TACS_Colab_Bundle.zip')
PROJECT_DIR = Path('/content/BTL_XLA')
DRIVE_OUTPUTS_ROOT = Path('/content/drive/MyDrive/TACS_outputs')
SEEDS = [42]
BASELINES = ['no_context', 'random', 'similarity', 'tacs', 'two_stage']
EPOCHS = 100
BATCH_SIZE = 4  # Giữ giống run local đang dở để so sánh công bằng.
NUM_WORKERS = 2
print('Configured seeds:', SEEDS)
print('Drive outputs:', DRIVE_OUTPUTS_ROOT)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
assert BUNDLE_PATH.is_file(), f'Không tìm thấy bundle: {BUNDLE_PATH}'
print('Bundle size (MB):', round(BUNDLE_PATH.stat().st_size / 1024**2, 1))

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile

# Source chạy trên SSD tạm của Colab; chỉ output/checkpoint đặt trên Drive.
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    archive.extractall(PROJECT_DIR)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_DIR)],
    check=True,
)
os.chdir(PROJECT_DIR)
print('Installed project from:', PROJECT_DIR)

In [ ]:
# Nạp checkpoint bootstrap vào Drive đúng một lần, không ghi đè checkpoint mới hơn.
bootstrap_root = PROJECT_DIR / 'outputs'
copied = []
if bootstrap_root.is_dir():
    for source in bootstrap_root.rglob('*'):
        if not source.is_file():
            continue
        destination = DRIVE_OUTPUTS_ROOT / source.relative_to(bootstrap_root)
        if not destination.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            copied.append(str(destination))
print(f'Bootstrap files copied: {len(copied)}')
for path in copied:
    print(' -', path)

In [ ]:
import torch
assert torch.cuda.is_available(), 'Colab chưa bật GPU: Runtime → Change runtime type → GPU'
gpu = torch.cuda.get_device_properties(0)
print('PyTorch:', torch.__version__)
print('GPU:', gpu.name)
print('VRAM (GB):', round(gpu.total_memory / 1024**3, 1))
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
# HF_TOKEN là tùy chọn; thêm trong Colab Secrets để tải model ổn định hơn.
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('HF_TOKEN enabled')
else:
    print('HF_TOKEN not set; public download will still be attempted')

In [ ]:
# CIFAR-100 được tải vào SSD tạm mỗi runtime; split seed 42 nằm trong bundle.
from tacs.data import build_cifar100
datasets = build_cifar100(
    root=PROJECT_DIR / 'data',
    split_file=PROJECT_DIR / 'data/splits/cifar100_seed42.json',
    download=True,
)
print('CIFAR-100 ready:', {
    'query_train': len(datasets.query_train),
    'candidate_pool': len(datasets.candidate_pool),
    'validation': len(datasets.validation),
    'test': len(datasets.test),
})

In [ ]:
# Xem lệnh trước khi train.
base_command = [
    sys.executable, '-m', 'tacs.multiseed',
    '--dataset', 'cifar100',
    '--backbone', 'dinov3_vits16',
    '--device', 'cuda',
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--data-root', str(PROJECT_DIR / 'data'),
    '--split-file', str(PROJECT_DIR / 'data/splits/cifar100_seed42.json'),
    '--outputs-root', str(DRIVE_OUTPUTS_ROOT),
    '--seeds', *map(str, SEEDS),
    '--baselines', *BASELINES,
]
subprocess.run([*base_command, '--plan-only'], check=True)

In [ ]:
# Cell train chính. Có thể chạy lại sau khi runtime bị ngắt; trainer tự resume.
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
subprocess.run(base_command, check=True, env=env)

## Chuyển sang seed tiếp theo

Khi seed 42 hoàn tất, đổi `SEEDS = [43]` trong cell cấu hình rồi chạy lại từ cell xem lệnh. Làm tương tự với seed 44. Runner tự bỏ qua baseline hoàn chỉnh và tiếp tục baseline đang dở.

Kết quả tổng hợp nằm trong `MyDrive/TACS_outputs/cifar100/dinov3_vits16/multiseed_summary.*`.